# Section 1: Setup & Data Loading


In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from semopy import Model

In [ ]:
df = pd.read_excel("df.xlsx")

df['MET_log'] = np.log(df['MET_total'] + 1)

# Section 2: Descriptive Statistics


In [ ]:
variables = {
    'MET': 'MET_total',
    'Financial_Security': 'Q16',
    'Screen_Time': 'screen_time'
}

desc_results = []
for var_name, col_name in variables.items():
    data = df[col_name].dropna()
    desc_results.append({
        'Variable': var_name,
        'M': round(data.mean(), 2),
        'SD': round(data.std(), 2),
        'Skewness': round(data.skew(), 2),
        'Kurtosis': round(data.kurtosis(), 2)
    })

desc_table = pd.DataFrame(desc_results)

# Section 3: Cronbach's Alpha


In [ ]:
for col in ["Q12d", "Q12e", "Q12g", "Q12h"]:
    df[col + "_r"] = 6 - df[col]

for col in ["Q13b", "Q13d"]:
    df[col + "_r"] = 6 - df[col]

scale_items = {
    "SWLS": ["Q11a", "Q11b", "Q11c", "Q11d", "Q11e"],
    "PSS-10": ["Q12a", "Q12b", "Q12c", "Q12f", "Q12d_r", "Q12e_r", "Q12g_r", "Q12h_r", 'Q12i', "Q12j"],
    "BRS": ["Q13a", "Q13c", "Q13e", "Q13b_r", "Q13d_r"],
    "UCLA-3": ["Q14a", "Q14b", "Q14c"],
    "BAS-2": ["Q15a", "Q15b", "Q15c"]
}

def cronbach_alpha(items_df):
    items = items_df.dropna()
    k = items.shape[1]
    if k < 2:
        return np.nan
    item_vars = items.var(axis=0, ddof=1)
    total_var = items.sum(axis=1).var(ddof=1)
    return (k / (k - 1)) * (1 - item_vars.sum() / total_var)

alpha_results = []
for scale_name, items in scale_items.items():
    alpha = cronbach_alpha(df[items])
    n_items = len(items)
    alpha_results.append({
        'Scale': scale_name,
        'Items': n_items,
        'α': round(alpha, 3)
    })

alpha_table = pd.DataFrame(alpha_results)

# Section 3: Scale Construction & Descriptive Statistics


In [ ]:
# Create scale scores by averaging items
for scale_name, items in scale_items.items():
    df[scale_name] = df[items].mean(axis=1)

In [ ]:
variables = {
    'SWLS': 'SWLS',
    'PSS-10': 'PSS-10',
    'BAS-2': 'BAS-2',
    'BRS': 'BRS',
    'MET': 'MET_total',
    'Financial_Security': 'Q16',
    'Screen_Time': 'screen_time'
}

desc_results = []
for var_name, col_name in variables.items():
    data = df[col_name].dropna()
    desc_results.append({
        'Variable': var_name,
        'M': round(data.mean(), 2),
        'SD': round(data.std(), 2),
        'Skewness': round(data.skew(), 2),
        'Kurtosis': round(data.kurtosis(), 2)
    })

desc_table = pd.DataFrame(desc_results)

# Section 4: Correlation Matrix


In [ ]:
corr_vars = ["MET_total", "screen_time", "SWLS", "PSS-10", "BRS", "UCLA-3", "BAS-2"]
corr_df = pd.DataFrame({v: df[v] for v in corr_vars})

n = len(corr_vars)
r_matrix = pd.DataFrame(np.zeros((n, n)), index=corr_vars, columns=corr_vars)
p_matrix = pd.DataFrame(np.zeros((n, n)), index=corr_vars, columns=corr_vars)

for i in range(n):
    for j in range(n):
        r, p = stats.pearsonr(corr_df.iloc[:, i], corr_df.iloc[:, j])
        r_matrix.iloc[i, j] = r
        p_matrix.iloc[i, j] = p

def format_corr(r, p):
    if p < 0.001:
        return f"{r:.3f}***"
    elif p < 0.01:
        return f"{r:.3f}**"
    elif p < 0.05:
        return f"{r:.3f}*"
    else:
        return f"{r:.3f}"

corr_formatted = pd.DataFrame(index=corr_vars, columns=corr_vars, dtype=object)
for i in range(n):
    for j in range(n):
        if i == j:
            corr_formatted.iloc[i, j] = "—"
        else:
            corr_formatted.iloc[i, j] = format_corr(r_matrix.iloc[i, j], p_matrix.iloc[i, j])

# Section 5: Simultaneous Multiple Regression + VIF


In [ ]:
df_reg = df[["MET_log", "PSS-10", "BRS", "BAS-2", "screen_time", "Q16", "Q1", "SWLS"]].copy()
df_reg.columns = ["MET_log", "PSS-10", "BRS", "BAS-2", "Screen time", "Financial security", "Gender", "SWLS"]

df_reg["Gender"] = df_reg["Gender"] - 1

cont_vars = ["MET_log", "PSS-10", "BRS", "BAS-2", "Screen time", "Financial security"]
for col in cont_vars:
    df_reg[col] = df_reg[col] - df_reg[col].mean()

X = sm.add_constant(df_reg[cont_vars + ["Gender"]])
y = df_reg["SWLS"]
model_full = sm.OLS(y, X).fit()

vif_data = pd.DataFrame({
    "Variable": X.columns[1:],
    "VIF": [variance_inflation_factor(X.values, i+1) for i in range(len(X.columns)-1)]
})

# Section 6: Path Analysis + Bootstrap CIs


In [ ]:
from scipy.stats import zscore

sem_vars = ["MET_log", "BAS_2", "PSS_10", "FinSec", "SWLS"]
sem_df = pd.DataFrame({
    "MET_log": zscore(df["MET_log"]),
    "BAS_2": zscore(df["BAS-2"]),
    "PSS_10": zscore(df["PSS-10"]),
    "FinSec": zscore(df["Q16"]),
    "SWLS": zscore(df["SWLS"]),
})

model_spec = """
PSS_10 ~ MET_log + BAS_2
SWLS ~ MET_log + BAS_2 + PSS_10 + FinSec
"""

model_path = Model(model_spec)
model_path.fit(sem_df)

In [ ]:
sem_results = model_path.inspect()

direct_bas_swls = sem_results[(sem_results['lval'] == 'SWLS') & 
                               (sem_results['op'] == '~') & 
                               (sem_results['rval'] == 'BAS_2')]['Estimate'].values[0]

a_path = sem_results[(sem_results['lval'] == 'PSS_10') & 
                     (sem_results['op'] == '~') & 
                     (sem_results['rval'] == 'BAS_2')]['Estimate'].values[0]

b_path = sem_results[(sem_results['lval'] == 'SWLS') & 
                     (sem_results['op'] == '~') & 
                     (sem_results['rval'] == 'PSS_10')]['Estimate'].values[0]

indirect_bas_swls = a_path * b_path

total_bas_swls = direct_bas_swls + indirect_bas_swls

direct_pa_swls = sem_results[(sem_results['lval'] == 'SWLS') & 
                              (sem_results['op'] == '~') & 
                              (sem_results['rval'] == 'MET_log')]['Estimate'].values[0]

a_path_pa = sem_results[(sem_results['lval'] == 'PSS_10') & 
                        (sem_results['op'] == '~') & 
                        (sem_results['rval'] == 'MET_log')]['Estimate'].values[0]

indirect_pa_swls = a_path_pa * b_path
total_pa_swls = direct_pa_swls + indirect_pa_swls

In [ ]:
n_boot = 10000
np.random.seed(3)
n_obs = len(sem_df)

boot_results = {
    "PSS_10~MET_log": np.zeros(n_boot),
    "PSS_10~BAS_2": np.zeros(n_boot),
    "SWLS~MET_log": np.zeros(n_boot),
    "SWLS~BAS_2": np.zeros(n_boot),
    "SWLS~PSS_10": np.zeros(n_boot),
    "SWLS~FinSec": np.zeros(n_boot),
    "indirect_BAS": np.zeros(n_boot),
    "indirect_PA": np.zeros(n_boot),
}

for i in range(n_boot):
    idx = np.random.choice(n_obs, n_obs, replace=True)
    boot_sample = sem_df.iloc[idx].reset_index(drop=True)
    
    try:
        model_boot = Model(model_spec)
        model_boot.fit(boot_sample, obj='MLW')
        params = model_boot.inspect().set_index(['lval', 'op', 'rval'])
        
        boot_results["PSS_10~MET_log"][i] = params.loc[('PSS_10', '~', 'MET_log'), 'Estimate']
        boot_results["PSS_10~BAS_2"][i] = params.loc[('PSS_10', '~', 'BAS_2'), 'Estimate']
        boot_results["SWLS~MET_log"][i] = params.loc[('SWLS', '~', 'MET_log'), 'Estimate']
        boot_results["SWLS~BAS_2"][i] = params.loc[('SWLS', '~', 'BAS_2'), 'Estimate']
        boot_results["SWLS~PSS_10"][i] = params.loc[('SWLS', '~', 'PSS_10'), 'Estimate']
        boot_results["SWLS~FinSec"][i] = params.loc[('SWLS', '~', 'FinSec'), 'Estimate']
        
        a_path_bas = params.loc[('PSS_10', '~', 'BAS_2'), 'Estimate']
        b_path = params.loc[('SWLS', '~', 'PSS_10'), 'Estimate']
        boot_results["indirect_BAS"][i] = a_path_bas * b_path
        
        a_path_pa = params.loc[('PSS_10', '~', 'MET_log'), 'Estimate']
        boot_results["indirect_PA"][i] = a_path_pa * b_path
    except:
        pass

# Section 7: Mediation Model Comparison


In [ ]:
model_spec_constrained = """
PSS_10 ~ MET_log + BAS_2
SWLS ~ MET_log + PSS_10 + FinSec + 0*BAS_2
"""

model_spec_unconstrained = """
PSS_10 ~ MET_log + BAS_2
SWLS ~ MET_log + BAS_2 + PSS_10 + FinSec
"""

model_constrained = Model(model_spec_constrained)
model_constrained.fit(sem_df)

model_unconstrained = Model(model_spec_unconstrained)
model_unconstrained.fit(sem_df)

from semopy import calc_stats

stats_constrained = calc_stats(model_constrained)
stats_unconstrained = calc_stats(model_unconstrained)

chi2_constrained = stats_constrained.loc['Value', 'chi2']
chi2_unconstrained = stats_unconstrained.loc['Value', 'chi2']
df_constrained = stats_constrained.loc['Value', 'DoF']
df_unconstrained = stats_unconstrained.loc['Value', 'DoF']

delta_chi2 = chi2_constrained - chi2_unconstrained
delta_df = df_constrained - df_unconstrained

from scipy.stats import chi2
p_value_chi2 = 1 - chi2.cdf(delta_chi2, delta_df)

In [ ]:
n_boot = 5000
np.random.seed(42)
boot_direct = np.zeros(n_boot)

for i in range(n_boot):
    idx = np.random.choice(n_obs, n_obs, replace=True)
    boot_sample = sem_df.iloc[idx].reset_index(drop=True)
    
    try:
        model_boot = Model(model_spec_unconstrained)
        model_boot.fit(boot_sample, obj='MLW')
        params = model_boot.inspect().set_index(['lval', 'op', 'rval'])
        boot_direct[i] = params.loc[('SWLS', '~', 'BAS_2'), 'Estimate']
    except:
        pass

valid_direct = boot_direct[boot_direct != 0]
ci_low, ci_high = np.percentile(valid_direct, [2.5, 97.5])

# Section 8: BRS Suppression Analysis


In [ ]:
valid_data = df[['BRS', 'SWLS']].dropna()
r_swls, p_swls = stats.pearsonr(valid_data['BRS'], valid_data['SWLS'])
n_swls = len(valid_data)

z = np.arctanh(r_swls)
se = 1 / np.sqrt(n_swls - 3)
ci_low_z = z - 1.96 * se
ci_high_z = z + 1.96 * se
ci_low_swls = np.tanh(ci_low_z)
ci_high_swls = np.tanh(ci_high_z)

valid_data_pss = df[['BRS', 'PSS-10']].dropna()
r_pss, p_pss = stats.pearsonr(valid_data_pss['BRS'], valid_data_pss['PSS-10'])
n_pss = len(valid_data_pss)

z_pss = np.arctanh(r_pss)
se_pss = 1 / np.sqrt(n_pss - 3)
ci_low_z_pss = z_pss - 1.96 * se_pss
ci_high_z_pss = z_pss + 1.96 * se_pss
ci_low_pss = np.tanh(ci_low_z_pss)
ci_high_pss = np.tanh(ci_high_z_pss)

In [ ]:
df_supp = df[["PSS-10", "BAS-2", "MET_log", "BRS", "Q16", "Q1", "screen_time", "SWLS"]].dropna().copy()
df_supp.columns = ["PSS10", "BAS2SF", "MET_log", "BRS", "FinSec", "Gender", "ScreenTime", "SWLS"]

df_supp["Gender"] = df_supp["Gender"] - 1

cont_vars_supp = ["PSS10", "BAS2SF", "MET_log", "BRS", "FinSec", "ScreenTime"]
for col in cont_vars_supp:
    df_supp[col] = df_supp[col] - df_supp[col].mean()

X_full = df_supp[['PSS10', 'BAS2SF', 'MET_log', 'BRS', 'FinSec', 'Gender', 'ScreenTime']]
y = df_supp['SWLS']
X_full_const = sm.add_constant(X_full)
model_A = sm.OLS(y, X_full_const).fit()

X_reduced = df_supp[['PSS10', 'BAS2SF', 'MET_log', 'FinSec', 'Gender', 'ScreenTime']]
X_reduced_const = sm.add_constant(X_reduced)
model_B = sm.OLS(y, X_reduced_const).fit()

PSS10_change = model_B.params['PSS10'] - model_A.params['PSS10']
PSS10_pct_change = (PSS10_change / model_A.params['PSS10']) * 100

# Section 9: Automated Results Verification

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from semopy import Model
from scipy.stats import zscore

df = pd.read_excel("/home/evababnik/my_data_project/my_data_project/notebooks/vojko mar 2026/df.xlsx")

df['MET_log'] = np.log(df['MET_total'] + 1)

for col in ["Q12d", "Q12e", "Q12g", "Q12h"]:
    df[col + "_r"] = 6 - df[col]
for col in ["Q13b", "Q13d"]:
    df[col + "_r"] = 6 - df[col]

scale_items = {
    "SWLS": ["Q11a", "Q11b", "Q11c", "Q11d", "Q11e"],
    "PSS-10": ["Q12a", "Q12b", "Q12c", "Q12f", "Q12d_r", "Q12e_r", "Q12g_r", "Q12h_r", 'Q12i', "Q12j"],
    "BRS": ["Q13a", "Q13c", "Q13e", "Q13b_r", "Q13d_r"],
    "UCLA-3": ["Q14a", "Q14b", "Q14c"],
    "BAS-2": ["Q15a", "Q15b", "Q15c"]
}

for scale_name, items in scale_items.items():
    df[scale_name] = df[items].mean(axis=1)

sem_df = pd.DataFrame({
    "MET_log": zscore(df["MET_log"]),
    "BAS_2": zscore(df["BAS-2"]),
    "PSS_10": zscore(df["PSS-10"]),
    "FinSec": zscore(df["Q16"]),
    "SWLS": zscore(df["SWLS"]),
})

model_spec_constrained = """
PSS_10 ~ MET_log + BAS_2
SWLS ~ MET_log + PSS_10 + FinSec + 0*BAS_2
"""

model_spec_unconstrained = """
PSS_10 ~ MET_log + BAS_2
SWLS ~ MET_log + BAS_2 + PSS_10 + FinSec
"""

model_constrained = Model(model_spec_constrained)
model_constrained.fit(sem_df)

model_unconstrained = Model(model_spec_unconstrained)
model_unconstrained.fit(sem_df)

attrs = [attr for attr in dir(model_constrained) if not attr.startswith('_')]

for attr in ['loss', 'objective', 'obj', 'fmin', 'result']:
    if hasattr(model_constrained, attr):
        pass
        
for attr in ['_loss', '_obj', '_fmin']:
    if hasattr(model_constrained, attr):
        val = getattr(model_constrained, attr)

In [ ]:
from semopy import calc_stats

stats_constrained = calc_stats(model_constrained)

stats_unconstrained = calc_stats(model_unconstrained)

if isinstance(stats_constrained, dict):
    pass
elif isinstance(stats_constrained, pd.DataFrame) or isinstance(stats_constrained, pd.Series):
    pass

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from semopy import Model, calc_stats
from scipy.stats import zscore

df = pd.read_excel("/home/evababnik/my_data_project/my_data_project/notebooks/vojko mar 2026/df.xlsx")
df['MET_log'] = np.log(df['MET_total'] + 1)

for col in ["Q12d", "Q12e", "Q12g", "Q12h"]:
    df[col + "_r"] = 6 - df[col]
for col in ["Q13b", "Q13d"]:
    df[col + "_r"] = 6 - df[col]

scale_items = {
    "SWLS": ["Q11a", "Q11b", "Q11c", "Q11d", "Q11e"],
    "PSS-10": ["Q12a", "Q12b", "Q12c", "Q12f", "Q12d_r", "Q12e_r", "Q12g_r", "Q12h_r", 'Q12i', "Q12j"],
    "BRS": ["Q13a", "Q13c", "Q13e", "Q13b_r", "Q13d_r"],
    "UCLA-3": ["Q14a", "Q14b", "Q14c"],
    "BAS-2": ["Q15a", "Q15b", "Q15c"]
}

for scale_name, items in scale_items.items():
    df[scale_name] = df[items].mean(axis=1)

variables = {
    'SWLS': 'SWLS',
    'PSS-10': 'PSS-10',
    'BAS-2': 'BAS-2',
    'BRS': 'BRS',
    'MET': 'MET_total',
    'Financial_Security': 'Q16',
    'Screen_Time': 'screen_time'
}

desc_results = []
for var_name, col_name in variables.items():
    data = df[col_name].dropna()
    desc_results.append({
        'Variable': var_name,
        'M': round(data.mean(), 2),
        'SD': round(data.std(), 2),
        'Skewness': round(data.skew(), 2),
        'Kurtosis': round(data.kurtosis(), 2)
    })

desc_table = pd.DataFrame(desc_results)

def cronbach_alpha(items_df):
    items = items_df.dropna()
    k = items.shape[1]
    if k < 2:
        return np.nan
    item_vars = items.var(axis=0, ddof=1)
    total_var = items.sum(axis=1).var(ddof=1)
    return (k / (k - 1)) * (1 - item_vars.sum() / total_var)

alpha_results = []
for scale_name, items in scale_items.items():
    alpha = cronbach_alpha(df[items])
    n_items = len(items)
    alpha_results.append({
        'Scale': scale_name,
        'Items': n_items,
        'α': round(alpha, 3)
    })

alpha_table = pd.DataFrame(alpha_results)